<div style="background: linear-gradient(90deg, #0b6623 0%, #2e8b57 50%, #98fb98 100%); padding: 20px; border-radius: 12px; text-align: center;">
  <h1 style="color: white; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial; font-weight: 700; font-size: 2.2rem;">
    Lab 2
  </h1>
</div>

Implement and train Softmax Regression with mini-batch SGD and early stopping.

The expected outcome.
* Implement Softmax Regression Model.
* Implement mini-batch SGD.
* The training should support early stopping.
* Train and evaluate the model with cross-validation. The evaluation metric is the *accuracy*.
* Retrain the model with early stopping.


**DO NOT USE SKLEARN**

In [1]:
import numpy as np
import pandas as pd 

from sklearn import datasets
from sklearn.model_selection import StratifiedShuffleSplit

np.random.seed(42)

In [2]:

iris = datasets.load_iris()
X = iris["data"]
y = iris["target"]
df = pd.DataFrame({fname: values for fname, values in zip(iris["feature_names"], X.T)})
df["target"] = y

df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [3]:
m,n=X.shape
m,n

(150, 4)

## Your Code
You can start writing your code from here. Please don't modify any of the previous code.

In [4]:
def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [5]:
def mini_batch_GD(X, y, lr=0.01, iterations=1000, batch_size=20):
    m, n = X.shape
    k = len(np.unique(y))
    theta = np.random.randn(n, k) * 0.01
    
    y_onehot = np.zeros((m, k))
    y_onehot[np.arange(m), y] = 1

    for i in range(iterations):
     
        indices = np.random.permutation(m) 
        X_shuffled = X[indices]
        y_onehot_shuffled = y_onehot[indices]

      
        for start in range(0, m, batch_size):
            end = start + batch_size
            
            X_batch = X_shuffled[start:end]
            y_batch = y_onehot_shuffled[start:end]
            
            
            batch_len = len(X_batch)
            
          
            scores = X_batch @ theta
            probs = softmax(scores)
            
          
            gradient = X_batch.T @ (probs - y_batch) / batch_len
            theta -= lr * gradient
        

            

        if i % 100 == 0:
            scores = X @ theta
            probs = softmax(scores)
            loss = -np.mean(np.sum(y_onehot * np.log(probs + 1e-15), axis=1))
            print(f"Iteration {i}, Loss: {loss:.4f}")

            if loss < 0.01:
                break



            
    return theta

Using the following cell to train and evaluate your model.

In [6]:
split = StratifiedShuffleSplit(n_splits=3, test_size=0.2, random_state=42)
for train_index, test_index in split.split(df, df["target"]):
    strat_train_set = df.loc[train_index]
    strat_test_set = df.loc[test_index]

    # Prepare features and labels
    X_train = strat_train_set.drop("target", axis=1)
    y_train = strat_train_set["target"]
    X_test = strat_test_set.drop("target", axis=1)
    y_test = strat_test_set["target"]

    # Train the model
    theta = mini_batch_GD(X_train.values, y_train.values)
    
    

Iteration 0, Loss: 1.0497
Iteration 100, Loss: 0.4243
Iteration 200, Loss: 0.3364
Iteration 300, Loss: 0.2864
Iteration 400, Loss: 0.2529
Iteration 500, Loss: 0.2289
Iteration 600, Loss: 0.2107
Iteration 700, Loss: 0.1962
Iteration 800, Loss: 0.1847
Iteration 900, Loss: 0.1752
Iteration 0, Loss: 1.0343
Iteration 100, Loss: 0.4306
Iteration 200, Loss: 0.3430
Iteration 300, Loss: 0.2924
Iteration 400, Loss: 0.2580
Iteration 500, Loss: 0.2325
Iteration 600, Loss: 0.2132
Iteration 700, Loss: 0.1982
Iteration 800, Loss: 0.1856
Iteration 900, Loss: 0.1755
Iteration 0, Loss: 1.0502
Iteration 100, Loss: 0.4225
Iteration 200, Loss: 0.3340
Iteration 300, Loss: 0.2819
Iteration 400, Loss: 0.2476
Iteration 500, Loss: 0.2229
Iteration 600, Loss: 0.2041
Iteration 700, Loss: 0.1894
Iteration 800, Loss: 0.1775
Iteration 900, Loss: 0.1677


In [7]:
def predict(X, theta):
    scores = X @ theta
    probs = softmax(scores)
    return np.argmax(probs, axis=1)

In [9]:
# Evaluate the model
predictions = predict(X_test.values, theta)
accuracy = np.mean(predictions == y_test.values)
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 90.00%
